## DSAN 6000 Homework 5: "Freezing" Python Computations with Polars

> **Note**: If you run into RAM overflow issues at any point in the assignment (which will be indicated by, e.g., the kernel or the Python process being killed), you can switch from using `s3://dsan6000-data/acled_events.parquet` as the main data file to `s3://dsan6000-data/acled_civilian_events.parquet` instead.
>
> The latter file is about 6x smaller (since it only includes events in the ACLED database which involve civilian targets), and should prevent any RAM overflows. But, you should first try the assignment with the full `acled_events.parquet` file, to see the full extent of the memory savings!

## Overview

This assignment picks up right where you left off in HW4, where you gained some hands-on experience with the `.parquet` format: how it stores important metadata information in its file footer, and how it achieves compression by using the Run-Length Encoding (RLE) scheme to condense repeated data values.

Specifically, towards the end of the last assignment (Question 2.2 and Question 3), you may have noticed two things:

* (a) In **Question 2.2**, we had to employ an entirely new language – DuckDB's implementation of SQL – to carry out our data analysis, despite the fact that we were working with Python code in a `.py` file. Essentially, we used Python's triple-quoting feature to "smuggle in" code from an entirely separate language, by including that code between a starting and ending `"""`.
* (b) From this same perspective, **Question 3** may have felt even more bizarre: not only were we employing the same importing-non-Python-code-into-Python trick as in Q2.2, but this time we were also using it to query a **Python object**, the Pandas `DataFrame` that had been created in a previous code cell.

If you love SQL, and if you find yourself more easily thinking through data-analysis tasks via its **declarative** format, you can carry on using this hybrid SQL-queries-within-Python approach after this assignment (for example, in your **Final Project**❗️).

But, if you were disturbed by this mixing-and-matching of languages, **Polars** is here in HW5 to help you sleep a bit better! Because, as promised in the intro email sent out before the course even started, in this assignment you will learn how to take the [**eagerly**-evaluated] Pandas workflows you are already familiar with and "translate" them into **lazily**-evaluated Python code (as opposed to SQL code that you shove into Python via `"""`).

After a refresher/slightly-deeper-dive into the Parquet format in Part 1, the remaining parts will have you resuscitate your Pandas code from HW4 Q2.1 and then walk you through "translating" it into Polars code. But before you start, here and elsewhere, please keep the **core difference** between the two approaches in the front of your mind:

> ***Pandas*** allows you to instantly see the effects (e.g., in the output of a code cell) of applying a "piece" of computation to a "piece" of data, due to its **eager evaluation** approach
>
> ***Polars***, on the other hand, **separates computation from data**, allowing its underlying (Rust-based) **query optimizer** to **plan out in advance** the best way to carry out a given computation on a given dataset, before actually running it.

Finally, linking this assignment back to HW4 once more, note how your writing of DuckDB queries as Python strings *implicitly* achieved this separation of computation from data: you had to write a query from start to finish, as a self-contained unit, *before* asking the DuckDB engine (via `con.execute()`) to *apply it* to a particular piece of data. This is how you were then able to easily "move" this unit of computation from the `.py` file in Q2.2 querying an external S3 object to the `.ipynb` file in Q3 querying an in-memory `DataFrame`!

## Part 1: A Deeper Dive Into Parquet's RLE Compression and Row-Group Partitioning

In HW4 Q1, you opened up the "black box" of the `.parquet` format, explicitly looking into the metadata stored in its footer and its compression of the data within each column (with varying degrees of "success" in terms of its ability to compress different formats).

This question, partly as a refresher and partly as a deeper-dive, will first test your understanding of RLE Compression and then move onto the additional "dimension" available to Parquet for compression: the partition of a full dataset into separate `.parquet`-encoded row groups based on some important index of the rows (typically a **date**).

### Part 1.1: What Makes Columns More vs. Less Compressible?

The code cell with title `Q1.1-generate` below constructs a large-$N$ `DataFrame` called `q1_df`, wherein each column has a particular **distribution** of data values (making it more or less amenable to Parquet's specific compression techniques):

| Column Name | Distribution |
| ----------- | ------------ |
| `id`        | Integers 1 to $N$, sorted |
| `is_data`   | Always `True` |
| `tier`      | *Runs* of consecutive *identical values*: First $1/4$ of rows have value `A`, second $1/4$ have value `B`, and so on up to `D` |
| `category`  | 4 unique values (`A`, `B`, `C`, `D`), with 250,000 instances of each value (like `rank`), but **randomly shuffled** in this case: 250,000 indices randomly sampled without replacement for `A`, 250,000 indices randomly sampled without replacement for `B`, and so on up to `D`. |
| `rand_val`  | Each value is [pseudo-]randomly sampled from the uniform distribution $\mathcal{U}(0,1)$ |


After reading this table, run the *next* code cell to generate the `DataFrame` itself:

In [1]:
#| label: Q1.1-generate
import pandas as pd
import numpy as np
rng = np.random.default_rng(seed=6000)
N = 1_000_000

# id: 1 to N, increasing order
q1_df = pd.DataFrame({'id': range(N)})

# is_data: All values are identical (True)
q1_df['is_data'] = True

# rank: Column contains *runs* of consecutive identical values
q1_df['tier'] = 'A'
q1_df.iloc[250_000:500_000, -1] = 'B'
q1_df.iloc[500_000:750_000, -1] = 'C'
q1_df.iloc[750_000:N, -1] = 'D'

# category: four unique values, but, permuted randomly across rows
q1_df['category'] = rng.permuted(q1_df['tier'])

# rand_val: random uniform values in [0,1]
q1_df['rand_val'] = rng.random(size=N)

q1_df

Out[1]: 
            id  is_data tier category  rand_val
0            0     True    A        A  0.945182
1            1     True    A        D  0.683398
2            2     True    A        C  0.763598
3            3     True    A        A  0.661641
4            4     True    A        C  0.365353
...        ...      ...  ...      ...       ...
999995  999995     True    D        B  0.107236
999996  999996     True    D        B  0.968375
999997  999997     True    D        B  0.642580
999998  999998     True    D        C  0.337793
999999  999999     True    D        D  0.136377

[1000000 rows x 5 columns]


### Question 1.1a: Initial Guess

In the code cell below, **rank these columns** based on which you think will be most and least compressible when the data is exported in the `.parquet` format. That is, in Q1.1b below (*after* you make your guesses), you will compute the value of `comp_ratio`, the ratio of:

* (a) `memory_mb`: The memory required to store the column's values **in RAM, in the Pandas `DataFrame` `data_df`**, to
* (b) `comp_memory_mb`: The memory required to store the column's values **on disk, in a `.parquet` file**

So, before computing this ratio, mark your initial guesses: if you think this value will be highest for `rand_val` (i.e., that `rand_val` is the most compressible column), for example, then the Python string `"rand_val"` should come first in the list.

Note that this cell is graded **entirely based on completion, not correctness!** Meaning, you won't gain any benefit from updating your answer here after the ratios are computed below (and, it helps us to know how much to emphasize compressibility in class 😎)

In [2]:
#| label: Q1.1a-response
# Initial prediction: constant values and long runs should compress best;
# shuffled categories should still benefit from dictionary encoding.
compression_rank = ['is_data', 'tier', 'category', 'id', 'rand_val']

for rank_index, rank_col in enumerate(compression_rank):
    print(rank_index, rank_col)


0 is_data
1 tier
2 category
3 id
4 rand_val


### Question 1.1b: Computing Compression Ratios

Now to compute the ratios, to check the intuition you recorded in the previous question! The following code cell provides you with a `compute_pandas_memory()` function, that takes in a Pandas `DataFrame` (calling it `input_df`) and returns a new `DataFrame` with one row per column in `input_df`:

* `col_name` gives the name of the column within the passed-in `input_df`, and
* `memory_mb` represents the total amount of RAM Pandas devotes to storing the values within that column.

The provided code then applies this function to `q1_df` to produce and display `mem_df`, which contains the **first half** of the info you need here: the amount of memory **in RAM** required by **Pandas** for each column.

The provided code then **saves** `q1_df` to the `.parquet` format, as `q1_data.parquet`, using Pandas' default settings (for example, here we're not utilizing fancier features like custom row-partitioning, which you'll see in Part 1.2!).

Your job is to use the DuckDB library to obtain the amount of **disk space** required to store each column, after the columns have been **compressed** into `q1_data.parquet`. As mentioned in the comment within the `Q1.2-response` cell, this information should be **added to** the already-generated `mem_df` `DataFrame`, as two new columns (one for the compressed memory footprint, and another for the compression ratio).

*[Hint] The **DuckDB approach** to obtainin Parquet file metadata (shown during the in-class demo in Week 4) will produce multiple columns with slightly different compression-related info: the column named `total_compressed_size` is the desired measure here.*

In [3]:
#| label: Q1.2-init
# Part 1: Function computing memory usage by column
def compute_pandas_memory(input_df):
  result_df = (input_df.memory_usage(index=False) / (1024 ** 2)).to_frame().reset_index()
  result_df.columns = ['col_name','memory_mb']
  return result_df

# Part 2: Using this function to obtain RAM usage for q1_df
mem_df = compute_pandas_memory(q1_df)
display(mem_df)

# Part 3: Serializing q1_df to disk via Pandas' default settings
q1_df.to_parquet('q1_data.parquet')

,col_name,memory_mb
0,id,7.629395
1,is_data,0.953674
2,tier,8.583069
3,category,8.583069
4,rand_val,7.629395


In [4]:
#| label: Q1.1b-response
import duckdb

# Sum column chunks across every row group, converting bytes to MiB.
compressed_df = duckdb.sql("""
    SELECT path_in_schema AS col_name,
           SUM(total_compressed_size) / (1024.0 * 1024.0) AS comp_memory_mb
    FROM parquet_metadata('q1_data.parquet')
    GROUP BY path_in_schema
""").df()

comp_df = mem_df.merge(compressed_df, on='col_name', validate='one_to_one')
comp_df['comp_ratio'] = comp_df['memory_mb'] / comp_df['comp_memory_mb']
comp_df = comp_df.sort_values('comp_ratio', ascending=False).reset_index(drop=True)
comp_df


Out[4]: 
   col_name  memory_mb  comp_memory_mb   comp_ratio
0      tier   8.583069        0.002235  3839.590444
1   is_data   0.953674        0.008106   117.647059
2  category   8.583069        0.242607    35.378471
3        id   7.629395        4.078837     1.870483
4  rand_val   7.629395        7.891830     0.966746


### Part 1.2: Why Do We Call Parquet a "Hybrid" File Format?

Here, take a minute to reflect on your findings from Q1.1b before moving to the next code cell! You don't need to respond to these explicitly, just think about them in your head — they're written out here just to... force you to see them while you scroll to the next part 😜

**Did the *ordering* of columns, from most to least compressible, match your expectations?**

**Did the *magnitude* of the compression ratios match your expectations?**

For example, did you expect that the difference in compression ratios between the *sorted* `rank` column and the *unsorted* `category` column would be so large? (Admittedly, I didn't think they'd be so large, when I was making the assignment!)

**How might these ratios *scale* for larger and larger datasets?**

This question raises some tricky issues, that we're going to address below! The problem is that, when the data files get *too* large you may want to **split** them into multiple pieces — for example, in class we've discussed a few times how files relating to North America might be kept on servers in North America, while files relating to Europe might be kept on servers in the EU, especially given the stricter GDPR **data privacy regulations** operating in the EU.

The difficulty comes from the fact that, sadly, any naïve "chopping" of a given `.parquet` file at some random point might ruin the entire compression scheme! For example, if you have an RLE-encoded column like the `is_data` column here, and you decide to "chop" the `demo.parquet` file at row 800,000 (to move the last 200,000 rows into a separate file called `demo_2.parquet`), you would (at minimum) need to write updated metadata into the new file reflecting that it now only has 200,000 consecutive identical values.

It gets worse, however, since (as you saw when displaying the Parquet metadata in HW4) Parquet keeps track of more in-depth statistics, such as the minimum and maximum values within a given column, all of which would need to be recomputed to carry out such a "chopping" without destroying the benefits of the Parquet format.

Thus, in the next question, you'll see how the Parquet schema **also provides functionality to carry out this of "chopping" as efficiently as possible**, by leveraging explicitly-marked **row groups** as partition keys.

### Question 1.2: Hive-Partitioning a `.parquet` File By Row Groups

Your job in this question is to **leverage** the hybrid-ness of the Parquet format to **tackle** this problem, of how to best partition a large file into several smaller pieces.

Note that, in a real-world setting, this would be a fairly complex optimization, in the sense that the "best" partition would depend not only on compression ratio but also on e.g. **how often** the data is queried by filtering on some key within an organization.

Here, however, "best" is defined more simply: the choice of  **single** partition column (via the `partition_cols` argument to Pandas' `to_parquet()` function) that results in the lowest **total** disk space being taken up, summed across all `.parquet` files resulting from the partition.

Thus your task here has two parts:

* In the cell labeled `Q1.2a-response`, use Pandas to save `q1_df` in Hive-partitioned format, within a base output directory named `q1_partitioned`.
* Then, in the cell labeled `Q1.2b-response`, use the same approach you used in Q1.1 to obtain column-by-column disk space footprints (you can copy your code from the above cells), but this time compute `part_comp_ratio`, the ratio of RAM required by Pandas to store the values in memory to the **total** disk space required to store the values across all `.parquet` files.

In [5]:
#| label: Q1.2a-response
from pathlib import Path
from tempfile import TemporaryDirectory

# Compare the practical low-cardinality keys using complete file sizes,
# including file metadata. id and rand_val would create one directory per row.
partition_sizes = []
with TemporaryDirectory() as temp_dir:
    for column in ['is_data', 'tier', 'category']:
        output_dir = Path(temp_dir) / column
        q1_df.to_parquet(output_dir, partition_cols=[column])
        total_bytes = sum(p.stat().st_size for p in output_dir.rglob('*.parquet'))
        partition_sizes.append({'partition_key': column, 'total_bytes': total_bytes})

partition_sizes_df = pd.DataFrame(partition_sizes).sort_values('total_bytes')
display(partition_sizes_df)
partition_key = partition_sizes_df.iloc[0]['partition_key']
print(f'Chosen partition key: {partition_key}')
# Replace matching generated partitions so rerunning does not duplicate rows.
q1_df.to_parquet(
    'q1_partitioned',
    partition_cols=[partition_key],
    existing_data_behavior='delete_matching',
)


Chosen partition key: category


,partition_key,total_bytes
2,category,15513071
0,is_data,16043154
1,tier,16050838


In [6]:
#| label: Q1.2b-response
part_mem_df = compute_pandas_memory(q1_df)

partition_compressed_df = duckdb.sql("""
    SELECT path_in_schema AS col_name,
           SUM(total_compressed_size) / (1024.0 * 1024.0) AS part_comp_memory_mb
    FROM parquet_metadata('q1_partitioned/**/*.parquet')
    GROUP BY path_in_schema
""").df()

part_mem_df = part_mem_df.merge(
    partition_compressed_df, on='col_name', how='left', validate='one_to_one'
)
# Hive stores the partition key in directory names, not in column chunks.
# Its column-chunk footprint is zero; a finite compression ratio is undefined.
is_partition_key = part_mem_df['col_name'].eq(partition_key)
part_mem_df.loc[is_partition_key, 'part_comp_memory_mb'] = 0.0
part_mem_df['part_comp_ratio'] = (
    part_mem_df['memory_mb']
    / part_mem_df['part_comp_memory_mb'].replace(0, np.nan)
)
print(f'{partition_key}: stored in directory names; ratio shown as NaN.')
part_mem_df


category: stored in directory names; ratio shown as NaN.
Out[6]: 
   col_name  memory_mb  part_comp_memory_mb  part_comp_ratio
0        id   7.629395             5.458430         1.397727
1   is_data   0.953674             0.011768        81.037277
2      tier   8.583069             0.007842      1094.491062
3  category   8.583069             0.000000              NaN
4  rand_val   7.629395             9.258586         0.824035


## Part 2: Analyzing *Outside*-of-Memory Hive-Partitioned Data (In A Remote S3 Bucket)

Now that you know how to take advantage of Hive partitioning when creating and analyzing data in `.parquet` format, let's put that knowledge to work, using Polars to analyze a Hive-partitined dataset as efficiently as possible!

Specifically, we will carry out some data analysis tasks that will feel familiar to you from the prior practice you've had with Pandas `DataFrame`s. The new element here will be carrying out these workflows using `LazyFrame`s as lazily-evaluated substitutes for `DataFrame`s, and your job will be to become more and more comfortable with this approach.

The scary part will be that, as mentioned in class, **`LazyFrame`s don't actually contain any data in themselves!** They contain **"blueprints"** for the creation of `DataFrame`s from input data, based on chained-together **expressions** which "explain" to Polars the operations you're hoping to carry out on the data.

In this part, the goal is to simulate working with a dataset that is **too big to fit in your EC2 instance's memory**. I use "simulate" here because:

* Even if the dataset in its *current* form *does* happen to fit in your instance's memory...
* This is **event** data, where new rows are added each day...
* And we have no prior "guarantee" regarding **how many** rows might be added within a given day...

As historians have pointed out, for example, during political revolutions ["the developments of years are compressed into months and days"](https://www.google.com/books/edition/The_Proletarian_Revolution_in_Russia/3_MeAAAAMAAJ?hl=en&gbpv=1&pg=PA160&printsec=frontcover&dq=compressed%20days)... your task here (and throughout this class) is to write data-analysis pipelines that are **robust** and **scalable** enough to handle these potential **waves** of events!

For business-analytics data, similarly, you can imagine **news coverage** of your company suddenly leading to an exponential increase in page-visit events in the OLTP DB, such that the derived OLAP data that previously fit into RAM suddenly overflows your instance!

Being able to **develop with scalability in mind** is thus an immensely valuable skill in the 2026-and-beyond big-data world, and (as you'll see in Part 3), can be seamlessly "interleaved" with in-memory analytics by **detecting** when your data has been condensed enough to fit in RAM and then **caching** materialized `LazyFrames` locally.

### Question 2.1: Obtaining a Parquet Schema via a Polars `LazyFrame`

As mentioned earlier in the notebook, here you will be using the **same [ACLED event data](https://acleddata.com/)** you analyzed in HW4. This time, however, the remote data has been **Hive-partitioned**, with the key-specific (year-specific) `.parquet` files nested within an `acled_events` "folder" at the root of the `dsan6000-data` S3 bucket.

So, in the following code cell, construct a **Polars `LazyFrame`** (*not* a Polars `DataFrame`!) called `acled_lf`, ensuring that the data is **not** actually loaded into the RAM of your EC2 instance when this cell is run.

At the end of the cell, store the result from calling `acled_lf.collect_schema()` into a variable named `acled_schema`, and print the contents of this variable, to see the data types for each column in the remote Hive-partitioned data files:

In [7]:
#| label: Q2.1-response
import polars as pl
import shutil
from IPython.display import display

# The actual bucket prefix uses a hyphen: acled-events (not acled_events).
# Public data can be read anonymously without AWS credentials.
acled_lf = pl.scan_parquet(
    's3://dsan6000-data/acled-events/**/*.parquet',
    hive_partitioning=True,
    storage_options={'aws_region': 'us-east-1', 'aws_skip_signature': 'true'},
    credential_provider=None,
)
acled_schema = acled_lf.collect_schema()
print(acled_schema)

# The assignment allows graph code without Graphviz installed.
# Show the same optimized plan as text on machines without the dot executable.
def display_plan(lf):
    if shutil.which('dot'):
        display(lf.show_graph(plan_stage='ir'))
    else:
        print('Graphviz is unavailable; optimized query plan:')
        print(lf.explain(optimized=True))


Schema([('event_id_cnty', String), ('event_date', String), ('time_precision', Int64), ('disorder_type', String), ('event_type', String), ('sub_event_type', String), ('actor1', String), ('assoc_actor_1', String), ('inter1', String), ('actor2', String), ('assoc_actor_2', String), ('inter2', String), ('interaction', String), ('civilian_targeting', String), ('iso', Int64), ('region', String), ('country', String), ('admin1', String), ('admin2', String), ('admin3', String), ('location', String), ('latitude', Float64), ('longitude', Float64), ('geo_precision', Int64), ('source', String), ('source_scale', String), ('notes', String), ('fatalities', Int64), ('tags', String), ('timestamp', Int64), ('year', Int64)])


### Question 2.2: A Polars Expression for "Rows Involving Lebanon"

As mentioned in class, the syntax for creating Polars **expressions** takes some getting used to, but keep in mind: Becoming comfortable with this syntax means that you will have the power to **code exact equivalents for anything SQL can do**, while **utilizing a Pandas-esque set of Python functions**, instead of the separate language required by DuckDB!

So, in the following code cell,

1.  Construct a **Polars expression** that "encodes" the following statement:
    
    > *«The column named `country` is equal to the string value `'Lebanon'`»*
    
1.  Then, using the `.alias()` function that is available on all Polars expression objects (objects of type `polars.expr.expr.Expr` ...it's not a typo, `expr` repeats in the class hierarchy for some reason), give this expression the alias `'is_lebanon'`
1.  Finally, store the full expression object as a Python variable named `lbn_expr`.

Note that you are not **applying** this expression to any data yet! You are simply creating an expression that Polars can use *later*, to plan out and then execute its queries with respect to a particular input dataset.

To show that you have constructed the expression correctly, the provided code then prints two things:

1.  The **type** of `lbn_exr`, and
1.  The `lbn_expr` object itself

In [8]:
#| label: Q2.2-response

# Your code here: Replace the None with a Polars expression object
lbn_expr = (pl.col("country") == "Lebanon").alias('is_lebanon')

# (For graders to easily see the type and contents of your expression object)
print(type(lbn_expr))
print(lbn_expr)

<class 'polars.expr.expr.Expr'>
[(col("country")) == ("Lebanon")].alias("is_lebanon")


### Quick Interlude: Installing `graphviz` On Your EC2 Instance

Since the remaining questions involve displaying Polars' **optimized query plans** as graphs, you'll need to install the `graphviz` library on your local EC2 instance by executing

```
sudo apt install graphviz
```

within the Integrated Terminal in VSCode.

If this command works as intended, you should be able to wrap calls to `<LazyFrame>.show_graph(plan_stage='ir')` in the `display()` function, to see a visual representation of the query plan.

If you are unable to get `graphviz` working, this isn't fatal for the remaining questions: as long as you *write* the lines of code that *would* display these graphs as intended on an instance with a working `graphviz` library, that is sufficient for full points.

### Question 2.3: *Applying* the `lbn_expr` Expression to `acled_lf`

Now that the Polars expression you wrote is available for use, **apply it** to the ACLED data as follows:

1.  First construct a new `LazyFrame` called `lbn_lf`, containing the **plan** to apply `lbn_expr` as a **filter** on the full set of rows in `acled_df`.
1.  Use the `.show_graph(plan_stage='ir')` function available on all `LazyFrame`s to display the **optimized query plan** generated by Polars for this `LazyFrame`. Make sure to wrap this inside a call to `display()` so that the cell always displays the graph when executed.
1.  Finally, write a line of code that produces a Polars `pl.DataFrame` (*not* a `LazyFrame`) called `lbn_head_pl`, containing the **materialized first 5 rows** that result from Polars actually **carrying out** the instructions contained in `lbn_lf`. End the code cell by just displaying these 5 materialized rows.

*[Hint] Make sure you are utilizing the **lazy** functionality of `lbn_lf`! Only **5** rows worth of data should actually be loaded into Python's memory here, stored as `lbn_head_pl`, and then displayed*

In [9]:
#| label: Q2.3-response
lbn_lf = acled_lf.filter(lbn_expr)
display_plan(lbn_lf)

# Limit before collect: materialize only the five requested rows.
lbn_head_pl = lbn_lf.head(5).collect()
lbn_head_pl


Graphviz is unavailable; optimized query plan:
Parquet SCAN [s3://dsan6000-data/acled-events/year=1997/24796cfe7d3746768047e4e7669975e9-0.parquet, ... 28 other sources]
PROJECT */31 COLUMNS
SELECTION: col("country") == "Lebanon"
ESTIMATED ROWS: 2781490
Out[9]: 
shape: (5, 31)
┌─────────────┬────────────┬─────────────┬─────────────┬───┬────────────┬──────┬────────────┬──────┐
│ event_id_cn ┆ event_date ┆ time_precis ┆ disorder_ty ┆ … ┆ fatalities ┆ tags ┆ timestamp  ┆ year │
│ ty          ┆ ---        ┆ ion         ┆ pe          ┆   ┆ ---        ┆ ---  ┆ ---        ┆ ---  │
│ ---         ┆ str        ┆ ---         ┆ ---         ┆   ┆ i64        ┆ str  ┆ i64        ┆ i64  │
│ str         ┆            ┆ i64         ┆ str         ┆   ┆            ┆      ┆            ┆      │
╞═════════════╪════════════╪═════════════╪═════════════╪═══╪════════════╪══════╪════════════╪══════╡
│ LBN6985     ┆ 2016-01-01 ┆ 1           ┆ Political   ┆ … ┆ 0          ┆ null ┆ 1603128565 ┆ 2016 │
│             ┆ 

### Question 2.4: How Many Events Per Year?

In the following code cell:

1.  Construct a `LazyFrame` called `yearly_lf`, which builds on the `acled_lf` `LazyFrame` you created earlier, but now represents a **plan** to:
    1.  Group the data by **`year`**,
    1.  Compute the number of observations per year, and
    1.  Sort the results by year in ascending order
2.  Use `.show_graph(plan_stage='ir')` (wrapped in a `display()` call) to display Polars' optimized version of the `yearly_lf` plan, and then
3.  Use `.collect()` to **materialize** `yearly_lf` into a Polars `pl.DataFrame` called `yearly_pl`, and display the contents of `yearly_pl` as the result of running the cell

In [10]:
#| label: Q2.4-response
yearly_lf = acled_lf.group_by('year').agg(pl.len().alias('event_count')).sort('year')
display_plan(yearly_lf)
yearly_pl = yearly_lf.collect()
yearly_pl


Graphviz is unavailable; optimized query plan:
SORT BY [col("year")]
  UNION[maintain_order: false]
    PLAN 0:
      AGGREGATE[maintain_order: false]
        [len().alias("event_count")] BY [col("year")]
        FROM
        Parquet SCAN [s3://dsan6000-data/acled-events/year=1997/24796cfe7d3746768047e4e7669975e9-0.parquet]
        PROJECT 1/31 COLUMNS
        SELECTION: col("year").alias("").is_in([Series[year].alias("")])
        ESTIMATED ROWS: 95914
    PLAN 1:
      AGGREGATE[maintain_order: false]
        [len().alias("event_count")] BY [col("year")]
        FROM
        Parquet SCAN [s3://dsan6000-data/acled-events/year=1998/24796cfe7d3746768047e4e7669975e9-0.parquet]
        PROJECT 1/31 COLUMNS
        SELECTION: col("year").alias("").is_in([Series[year].alias("")])
        ESTIMATED ROWS: 95914
    PLAN 2:
      AGGREGATE[maintain_order: false]
        [len().alias("event_count")] BY [col("year")]
        FROM
        Parquet SCAN [s3://dsan6000-data/acled-events/year=1999/24

### Question 2.5: What Proportion of the Yearly Events Are In Lebanon?

Now **combine** the approach you've used in the past two questions to compute a yearly count (sorted by year in ascending order) of the number of events **matching your `lbn_expr` expression** per year.

As in the previous cells, here you should start by creating a **plan** (a `LazyFrame` object), then display the **optimized** version of the plan, and finally **materialize** the results of the plan into a Polars `pl.DataFrame`, displaying its contents as the result of running the cell.

In [11]:
#| label: Q2.5-response
lbn_yearly_lf = (
    acled_lf.filter(lbn_expr)
    .group_by('year')
    .agg(pl.len().alias('lebanon_event_count'))
    .sort('year')
)
display_plan(lbn_yearly_lf)
lbn_yearly_pl = lbn_yearly_lf.collect()
lbn_yearly_pl


Graphviz is unavailable; optimized query plan:
SORT BY [col("year")]
  UNION[maintain_order: false]
    PLAN 0:
      AGGREGATE[maintain_order: false]
        [len().alias("lebanon_event_count")] BY [col("year")]
        FROM
        simple π 1/1 ["year"]
          Parquet SCAN [s3://dsan6000-data/acled-events/year=1997/24796cfe7d3746768047e4e7669975e9-0.parquet]
          PROJECT 2/31 COLUMNS
          SELECTION: (col("year").alias("").is_in([Series[year].alias("")]) & (col("country") == "Lebanon"))
          ESTIMATED ROWS: 95914
    PLAN 1:
      AGGREGATE[maintain_order: false]
        [len().alias("lebanon_event_count")] BY [col("year")]
        FROM
        simple π 1/1 ["year"]
          Parquet SCAN [s3://dsan6000-data/acled-events/year=1998/24796cfe7d3746768047e4e7669975e9-0.parquet]
          PROJECT 2/31 COLUMNS
          SELECTION: (col("year").alias("").is_in([Series[year].alias("")]) & (col("country") == "Lebanon"))
          ESTIMATED ROWS: 95914
    PLAN 2:
      AGGREG

## Part 3: Caching Intermediate Results in Local RAM

At the end of Part 2, you should have discovered that **ACLED only began tracking events in Lebanon starting in 2016!**

So, let's assume we've done some back-of-the-envelope calculations, and we have identified that the *subset* of records relating to Lebanon *does* indeed fit inside RAM (in our simulated working example here).

This means that we have "unlocked" a new stage in the data analysis! Although Polars does optimize the loading from S3 as much as possible, every Polars expression we have materialized thus far (by calling `collect()`) has had to re-read the data from scratch, from the remote S3 bucket into your EC2 instance's RAM.

However, with a subset of events that *do* fit entirely within RAM, we can move to a new stage where:

* We **materialize** the Lebanon events from the remote S3 bucket into the local RAM, but
* We then **still want to optimize** our queries on this locally-materialized data!

To achieve this, we can use some code that would seem strange if you didn't know that these were the goals, which you will see in the `Q3.2-new-plan-local-data` cell below.

But first, run the following code cell, which (if your `lbn_expr` expression works as intended) will finally **materialize** the full set of Lebanon-related events into the RAM of your EC2 instance, as a Polars `pl.DataFrame`:

In [12]:
#| label: Q3-materialize
lbn_pl = acled_lf.filter(lbn_expr).collect()
lbn_pl.head()

Out[12]: 
shape: (5, 31)
┌─────────────┬────────────┬─────────────┬─────────────┬───┬────────────┬──────┬────────────┬──────┐
│ event_id_cn ┆ event_date ┆ time_precis ┆ disorder_ty ┆ … ┆ fatalities ┆ tags ┆ timestamp  ┆ year │
│ ty          ┆ ---        ┆ ion         ┆ pe          ┆   ┆ ---        ┆ ---  ┆ ---        ┆ ---  │
│ ---         ┆ str        ┆ ---         ┆ ---         ┆   ┆ i64        ┆ str  ┆ i64        ┆ i64  │
│ str         ┆            ┆ i64         ┆ str         ┆   ┆            ┆      ┆            ┆      │
╞═════════════╪════════════╪═════════════╪═════════════╪═══╪════════════╪══════╪════════════╪══════╡
│ LBN6985     ┆ 2016-01-01 ┆ 1           ┆ Political   ┆ … ┆ 0          ┆ null ┆ 1603128565 ┆ 2016 │
│             ┆            ┆             ┆ violence    ┆   ┆            ┆      ┆            ┆      │
│ LBN449      ┆ 2016-01-01 ┆ 1           ┆ Political   ┆ … ┆ 0          ┆ null ┆ 1756770056 ┆ 2016 │
│             ┆            ┆             ┆ violence    ┆   ┆      

### Question 3.1a: Serializing the Materialized `pl.DataFrame()` to Disk

Note that at this point — temporarily! — we have a Polars `pl.DataFrame` that is *not* doing anything fancy, i.e., anything that a Pandas `DataFrame` couldn't also do.

For example, just as you used Pandas' `to_parquet()` function earlier, you can just as easily use **Polars'** `write_parquet()` function here, to serialize the now-in-local-RAM data to disk... and you will!

Write code in the following cell to carry this out, by saving **only the materialized contents of `lbn_pldf`** to a subdirectory called `lbn-data`, within your local EC2 working directory (which should be called `dsan6000-hw05`!)

Just as before, you should use the **Hive partitioning** scheme to "chop" the data into smaller `.parquet` files, each one corresponding to a year for which you have **Lebanon-specific** data (you will check this directly in Q3.1b below!)

In [13]:
#| label: Q3.1a-response
# lbn_pl is the materialized DataFrame from the supplied cell above.
# Replace matching year partitions so rerunning does not duplicate records.
lbn_pl.write_parquet(
    'lbn-data',
    use_pyarrow=True,
    pyarrow_options={
        'partition_cols': ['year'],
        'existing_data_behavior': 'delete_matching',
    },
)


### Question 3.1b: Checking the Locally-Serialized Data

To ensure everything has gone smoothly, in the following code cell, import Python's `os` library and use `os.listdir()` to print out a listing of the contents within `lbn-data`.

This will allow you (and the graders) to quickly see whether the local caching has gone as planned: since we know ACLED only has events for Lebanon from 2016 onwards, the subfolders within `lbn-data` should start from `year=2016`!

In [14]:
#| label: Q3.1b-response
import os

print('\n'.join(sorted(os.listdir('lbn-data'))))


year=2016
year=2017
year=2018
year=2019
year=2020
year=2021
year=2022
year=2023
year=2024
year=2025


### Question 3.2: Starting a *New "Plan"* for the Local Data

This is the key part of the local-caching procedure! Run the following code (for free points wohoo), which illustrates what this step looks like:

* Once we call `.lazy()` on a locally-materialized `pl.DataFrame`...
* From that point onwards, instead of always querying the **remote S3 bucket** at the start of every materialization step, Polars will instead optimize queries **relative to the data residing within the RAM** — specifically, the data within the RAM that we have now **materialized into the `pl.DataFrame` called `lbn_pldf`**

In [15]:
#| label: Q3.2-new-plan-local-data
lbn_lf = lbn_pl.lazy()
# This plan starts from the in-memory DataFrame, not the remote S3 files.
display_plan(lbn_lf)


Graphviz is unavailable; optimized query plan:
DF ["event_id_cnty", "event_date", "time_precision", "disorder_type", ...]; PROJECT */31 COLUMNS


### Question 3.3: Optimizing Queries on Locally-Cached Data

That call to `.lazy()`, on a `pl.DataFrame`, really does a lot of work here! To see the (much faster) local operation of Pandas on this cached data in action, in the following code cell:

1.  Write a Polars **expression**, with the *alias* `had_fatalities`, that "encodes" the predicate that
    
    > *«The value in the column named `fatalities` is strictly greater than 0»*

    And store this expression in a Python variable named `fatal_expr`
2.  Create a `LazyFrame` called `lbn_fatal_lf`, representing the **plan** to apply the expression `fatal_expr` to `lbn_lf` as a **filter**
3.  Display the optimized Intermediate Representation query plan for `lbn_lf`
4.  **Materialize** the query plan into a Polars `pl.DataFrame` named `lbn_fatal_pl`, and display the contents of this `pl.DataFrame` as the result of running the cell.

In [16]:
#| label: Q3.3-response
fatal_expr = (pl.col('fatalities') > 0).alias('had_fatalities')
lbn_fatal_lf = lbn_lf.filter(fatal_expr)
# Display the filtered plan so the fatality predicate is visible.
display_plan(lbn_fatal_lf)
lbn_fatal_pl = lbn_fatal_lf.collect()
lbn_fatal_pl


Graphviz is unavailable; optimized query plan:
FILTER col("fatalities") > 0
FROM
  DF ["event_id_cnty", "event_date", "time_precision", "disorder_type", ...]; PROJECT */31 COLUMNS
Out[16]: 
shape: (1_810, 31)
┌────────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬──────┐
│ event_id_c ┆ event_date ┆ time_preci ┆ disorder_t ┆ … ┆ fatalitie ┆ tags      ┆ timestamp ┆ year │
│ nty        ┆ ---        ┆ sion       ┆ ype        ┆   ┆ s         ┆ ---       ┆ ---       ┆ ---  │
│ ---        ┆ str        ┆ ---        ┆ ---        ┆   ┆ ---       ┆ str       ┆ i64       ┆ i64  │
│ str        ┆            ┆ i64        ┆ str        ┆   ┆ i64       ┆           ┆           ┆      │
╞════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪══════╡
│ LBN465     ┆ 2016-01-08 ┆ 1          ┆ Political  ┆ … ┆ 1         ┆ null      ┆ 175072182 ┆ 2016 │
│            ┆            ┆            ┆ violence   ┆   ┆           ┆           ┆ 2 

If all went as planned, you should notice a significant **speedup** in how long the above query took to materialize, relative to the cases in Part 2 where each execution of the query required **HTTP requests to AWS for the S3 bucket contents!**

And, this can be a helpful workflow to use throughout your career! 😉 You can:

* First take advantage of DuckDB and/or Polars' ability to optimize and then execute queries on **remote files** when first starting (e.g., in EDA mode), and then
* If you find that your workflow has "crunched" the remote data down enough that the remaining data can be stored entirely in local RAM, use `.collect()` to "extract" the data from the remote S3 bucket one last time, then `.lazy()` to **cache** the materialized `pl.DataFrame` and work with it in the exact same way (via Polars expressions).
